In [ ]:
#!/usr/bin/env python3
"""
Project 13 — E1 cross-model convergence, Borzoi leg (run on Kaggle, free GPU).
Companion to src/14_cross_model_convergence.py / docs/TASK_E1_cross_model.md.

WHY THIS RUNS ON KAGGLE, NOT IN THE PROJECT SANDBOX
-----------------------------------------------------
Borzoi (via the borzoi-pytorch package) needs a GPU; the project's compute
sandbox has none. Kaggle notebooks offer a free T4/P100 GPU (30h/week quota),
so this script is meant to be pasted into a new Kaggle notebook (Settings ->
Accelerator -> GPU T4 x2) and run there.

WHAT IT DOES
------------
1. Reads the exact gene sample src/14 already built
   (data/processed/cross_model_gene_sample.csv -- 400 (gene,lineage,is_candidate)
   pairs, ~393 unique genes) and the pre-built Borzoi track->lineage mapping
   (data/processed/borzoi_target_lineage_map.csv -- 63 ENCODE RNA-seq tracks
   across the 5 broad immune lineages: B, CD4-T, CD8-T, NK, monocyte;
   "activated/stimulated/treated/disease" tracks were excluded to stay
   resting-state-consistent with OneK1K, matching the project's established
   caveat that all measured data here is resting PBMCs).
2. For each unique gene: builds a 524,288bp (Borzoi's native window) hg38
   interval centered on the gene body, predicts RNA-seq tracks with Borzoi,
   averages the predicted coverage over the gene body for each of the 63
   mapped tracks, averages tracks within each lineage -> one predicted value
   per (gene, lineage).
3. z-scores each gene's 5-lineage predicted vector -> spec_pred_borzoi.
4. Writes borzoi_predictions.csv with columns [gene, lineage, spec_pred_borzoi]
   -- this is the ONLY file you need to bring back to the project folder, as
   data/processed/borzoi_predictions.csv. Then re-run on your own machine:
       python src/14_cross_model_convergence.py --analyze-only
   which auto-merges it and completes the 3-way (Decima/Borzoi/AlphaGenome)
   comparison + the window-size bonus check.

HOW TO USE ON KAGGLE
---------------------
1. New Kaggle Notebook, enable GPU (Settings -> Accelerator).
2. Upload data/processed/cross_model_gene_sample.csv and
   data/processed/borzoi_target_lineage_map.csv as a Kaggle Dataset (or paste
   their contents into a cell -- they're small, <50KB each), and also upload
   data/raw/metadata.h5ad's var table (gene coordinates) OR just the small
   CSV this script expects at GENE_COORDS_CSV (see below -- export it once
   from the project folder with the one-liner in that section).
3. Paste this whole file into a cell (or `!python this_file.py` after upload)
   and run. Expect ~1-3s/gene on a T4 -> ~15-20 min for 393 genes.
4. Download the output borzoi_predictions.csv, drop it into the project's
   data/processed/, and re-run src/14 with --analyze-only.

Notes / honesty
-----------------
- Borzoi's window (524,288bp / 524kb) is narrower than AlphaGenome's (1MB) --
  this is the intentional "bonus" comparison in src/14 (if the gap is
  distal-driven, AlphaGenome's larger window should show a SMALLER |gap| on
  candidates than Borzoi's).
- Borzoi's and AlphaGenome's immune RNA-seq tracks draw heavily on the SAME
  underlying ENCODE experiments (see the shared-data-source caveat in
  src/14's module docstring) -- Borzoi<->AlphaGenome agreement is therefore
  not fully independent evidence of a "paradigm-level" effect, more evidence
  against Decima-training-specific idiosyncrasy.
- If a gene's predicted values are constant across lineages (sd=0, e.g. very
  low expression everywhere), it is dropped from the z-scored output -- this
  is logged, not silently discarded.
"""
from __future__ import annotations
import json
import time
from pathlib import Path

import numpy as np
import pandas as pd

# ---------------------------------------------------------------
# Paths -- adjust for your Kaggle input dataset mount if needed.
# ---------------------------------------------------------------
INPUT_DIR = Path("/kaggle/input/datasets/ericdu847/project13-cross-model")  # wherever you upload the 3 files below
SAMPLE_CSV = INPUT_DIR / "cross_model_gene_sample.csv"
TARGET_MAP_CSV = INPUT_DIR / "borzoi_target_lineage_map.csv"
GENE_COORDS_CSV = INPUT_DIR / "gene_coords.csv"  # gene,chrom,strand,gene_start,gene_end
# To produce gene_coords.csv from the project folder (run locally, once, then upload):
#   python3 -c "
#   import h5py, pandas as pd
#   f = h5py.File('data/raw/metadata.h5ad','r')
#   gene_id = f['var']['gene_id'][:].astype(str)
#   chrom = f['var']['chrom']['categories'][:].astype(str)[f['var']['chrom']['codes'][:]]
#   strand = f['var']['strand']['categories'][:].astype(str)[f['var']['strand']['codes'][:]]
#   start = f['var']['gene_start'][:]; end = f['var']['gene_end'][:]
#   pd.DataFrame({'gene':gene_id,'chrom':chrom,'strand':strand,
#                 'gene_start':start,'gene_end':end}).to_csv('gene_coords.csv', index=False)
#   "

OUTPUT_CSV = Path("/kaggle/working/borzoi_predictions.csv")

LINEAGES = ["B", "CD4-T", "CD8-T", "NK", "monocyte"]
BORZOI_WINDOW = 524_288  # Borzoi's native input length


def log(msg):
    print(f"[borzoi] {msg}", flush=True)


def main():
    import os
    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            print(f"Found: {os.path.join(dirname, filename)}")
    log("pip install borzoi-pytorch (run once per Kaggle session)")
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "borzoi-pytorch"], check=True)

    import torch
    from borzoi_pytorch import Borzoi

    device = "cuda" if torch.cuda.is_available() else "cpu"
    log(f"device: {device}")
    if device == "cpu":
        log("WARNING: no GPU detected -- this will be slow. Enable GPU in Kaggle notebook settings.")

    sample = pd.read_csv(SAMPLE_CSV)
    target_map = pd.read_csv(TARGET_MAP_CSV)
    coords = pd.read_csv(GENE_COORDS_CSV).set_index("gene")
    log(f"{len(sample)} sample pairs, {sample['gene'].nunique()} unique genes, "
        f"{len(target_map)} Borzoi tracks mapped to {target_map['lineage'].nunique()} lineages")

    model = Borzoi.from_pretrained("johahi/borzoi-replicate-0").to(device).eval()

    # one-hot encode a genomic interval (A,C,G,T) -- Borzoi expects (4, L)
    BASE_IDX = {"A": 0, "C": 1, "G": 2, "T": 3}

    def fetch_and_onehot(chrom, start, end):
        """Fetch reference sequence via UCSC DAS/2bit is out of scope here --
        Kaggle has no bundled hg38 fasta by default. Use pyfaidx + a hg38
        fasta uploaded as a Kaggle dataset (e.g. the public 'GRCh38' dataset),
        OR fetch via the UCSC REST API for short-lived, small-scale use:
        https://api.genome.ucsc.edu/getData/sequence?genome=hg38;chrom=...;start=...;end=...
        """
        import urllib.request
        url = (f"https://api.genome.ucsc.edu/getData/sequence?genome=hg38;"
               f"chrom={chrom};start={start};end={end}")
        with urllib.request.urlopen(url, timeout=30) as resp:
            data = json.loads(resp.read())
        seq = data["dna"].upper()
        arr = np.zeros((4, len(seq)), dtype=np.float32)
        for i, b in enumerate(seq):
            if b in BASE_IDX:
                arr[BASE_IDX[b], i] = 1.0
        return arr

    genes = sorted(sample["gene"].unique())
    results = {}
    t0 = time.time()
    for i, gene in enumerate(genes):
        if gene not in coords.index:
            continue
        row = coords.loc[gene]
        center = (int(row["gene_start"]) + int(row["gene_end"])) // 2
        start = max(0, center - BORZOI_WINDOW // 2)
        end = start + BORZOI_WINDOW
        try:
            seq = fetch_and_onehot(row["chrom"], start, end)
            x = torch.from_numpy(seq).unsqueeze(0).to(device)
            with torch.no_grad():
                pred = model(x)  # (1, n_tracks, n_bins) per borzoi-pytorch's forward()
            pred = pred.squeeze(0).float().cpu().numpy()  # (n_tracks, n_bins)
            gene_body_frac_start = (int(row["gene_start"]) - start) / BORZOI_WINDOW
            gene_body_frac_end = (int(row["gene_end"]) - start) / BORZOI_WINDOW
            n_bins = pred.shape[1]
            b0 = max(0, int(gene_body_frac_start * n_bins))
            b1 = min(n_bins, int(gene_body_frac_end * n_bins) + 1)
            if b1 <= b0:
                b0, b1 = 0, n_bins  # gene body outside window bounds -> fall back to whole window
            lineage_vals = {}
            for lin in LINEAGES:
                idxs = target_map.loc[target_map.lineage == lin, "index"].tolist()
                idxs = [ix for ix in idxs if ix < pred.shape[0]]
                if not idxs:
                    continue
                lineage_vals[lin] = float(pred[idxs][:, b0:b1].mean())
            results[gene] = lineage_vals
        except Exception as e:  # noqa: BLE001
            log(f"  gene {gene} FAILED: {e}")
        if (i + 1) % 20 == 0 or (i + 1) == len(genes):
            elapsed = time.time() - t0
            log(f"  {i+1}/{len(genes)} done ({elapsed:.0f}s, ~{elapsed/(i+1)*(len(genes)-i-1):.0f}s remaining)")

    # z-score per gene across the 5 lineages
    recs = []
    n_dropped = 0
    for gene, lv in results.items():
        vals = np.array([lv.get(l, np.nan) for l in LINEAGES])
        if np.isnan(vals).any():
            n_dropped += 1
            continue
        sd = vals.std()
        if sd == 0:
            n_dropped += 1
            continue
        z = (vals - vals.mean()) / sd
        for li, lin in enumerate(LINEAGES):
            recs.append((gene, lin, z[li]))
    log(f"z-scored {len(results) - n_dropped}/{len(results)} genes ({n_dropped} dropped: incomplete or zero-variance)")

    out = pd.DataFrame(recs, columns=["gene", "lineage", "spec_pred_borzoi"])
    out.to_csv(OUTPUT_CSV, index=False)
    log(f"wrote {OUTPUT_CSV} ({len(out)} rows). Download this file and place it at "
        f"data/processed/borzoi_predictions.csv in the project folder, then run:\n"
        f"    python src/14_cross_model_convergence.py --analyze-only")


if __name__ == "__main__":
    main()